In [1]:
import pandas as pd
from sklearn.dummy import DummyClassifier

# Experimentação - Verificando o MVP

In [15]:
df_encoded = pd.read_csv("../data/processed/telco_customer_churn_eda_encoded.csv")
df_encoded.head()

from sklearn.model_selection import train_test_split
# Dividindo o dataset em features (X) e target (y)
X = df_encoded.drop(columns=['target', 'CLTV', 'Total Charges'])
y = df_encoded['target']
# Divida em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Dados brutos, sem tratamentos de desbalanceamento

In [16]:
import os
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

owner = os.getenv("GITHUB_ACTOR") or os.getenv("USER") or os.getenv("USERNAME")

# mlflow.set_experiment("telco_churn_classification")

# with mlflow.start_run(run_name="logistic_regression_baseline"):
#     mlflow.set_tag("owner", owner)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

# Predições
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Métricas
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)
test_f1 = f1_score(y_test, y_pred_test)
test_precision = precision_score(y_test, y_pred_test)
test_recall = recall_score(y_test, y_pred_test)

overfitting = train_accuracy - test_accuracy

# =====================
# Log no MLflow (quando quiser usar)
# =====================
# mlflow.log_metric("train_accuracy", train_accuracy)
# mlflow.log_metric("test_accuracy", test_accuracy)
# mlflow.log_metric("test_f1_score", test_f1)
# mlflow.log_metric("test_precision", test_precision)
# mlflow.log_metric("test_recall", test_recall)
# mlflow.log_metric("overfitting", overfitting)

print(f"=== LOGISTIC REGRESSION (CHURN) ===")
print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy:  {test_accuracy:.4f}")
print(f"Test F1 Score:  {test_f1:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall:    {test_recall:.4f}")
print(f"Overfitting:    {overfitting:.4f}")

=== LOGISTIC REGRESSION (CHURN) ===
Train Accuracy: 0.8170
Test Accuracy:  0.8098
Test F1 Score:  0.6278
Test Precision: 0.7063
Test Recall:    0.5650
Overfitting:    0.0072


## Iniciando o tratamento do desbalanceamento utilizando Class_weight

In [10]:
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

mlflow.set_experiment("telco_churn_classification")

with mlflow.start_run(run_name="logistic_regression_balanced"):

    model = LogisticRegression(
        random_state=42,
        max_iter=1000,
        class_weight='balanced'
    )

    model.fit(X_train, y_train)

    # Predições
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # Métricas
    train_accuracy = accuracy_score(y_train, y_pred_train)
    test_accuracy = accuracy_score(y_test, y_pred_test)
    test_f1 = f1_score(y_test, y_pred_test)
    test_precision = precision_score(y_test, y_pred_test)
    test_recall = recall_score(y_test, y_pred_test)

    # Log no MLflow
    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("test_f1_score", test_f1)
    mlflow.log_metric("test_precision", test_precision)
    mlflow.log_metric("test_recall", test_recall)

    # Overfitting
    overfitting = train_accuracy - test_accuracy
    mlflow.log_metric("overfitting", overfitting)

    print(f"=== LOGISTIC REGRESSION (BALANCED) ===")
    print(f"Train Accuracy: {train_accuracy:.4f}")
    print(f"Test Accuracy:  {test_accuracy:.4f}")
    print(f"Test F1 Score:  {test_f1:.4f}")
    print(f"Test Precision: {test_precision:.4f}")
    print(f"Test Recall:    {test_recall:.4f}")
    print(f"Overfitting:    {overfitting:.4f}")

=== LOGISTIC REGRESSION (BALANCED) ===
Train Accuracy: 0.7616
Test Accuracy:  0.7551
Test F1 Score:  0.6547
Test Precision: 0.5459
Test Recall:    0.8175
Overfitting:    0.0065


c:\Users\Patrick\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Devido ao desbalanceamento das classes (73% vs 27%), foi aplicada a estratégia de class_weight='balanced' na Regressão Logística, visando melhorar a capacidade do modelo em identificar clientes com churn. A abordagem resultou em aumento do recall, tornando o modelo mais sensível à classe minoritária.

| Métrica     | Baseline | Balanced | Mudança       |
| ----------- | -------- | -------- | ------------- |
| Accuracy    | 0.8098   | 0.7551   | ⬇️ (esperado) |
| Precision   | 0.7063   | 0.5459   | ⬇️            |
| Recall      | 0.5650   | 0.8175   | 🚀⬆️ MUITO    |
| F1 Score    | 0.6278   | 0.6547   | ⬆️            |
| Overfitting | 0.0072   | 0.0065   | ✅ ok          |

O modelo baseline apresentou bom desempenho geral, porém com recall limitado na identificação de churn.
Após aplicar class_weight='balanced', houve um aumento significativo no recall (de 56% para 81%), tornando o modelo mais eficaz na detecção de clientes propensos a churn, ainda que com redução na precisão — um trade-off aceitável para o contexto de negócio.

## Vamos utilizar o SMOTE para comparar a eficácia do modelo em comparação ao class_weight

In [17]:
import os
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from imblearn.over_sampling import SMOTE

owner = os.getenv("GITHUB_ACTOR") or os.getenv("USER") or os.getenv("USERNAME")

# mlflow.set_experiment("telco_churn_classification")

# with mlflow.start_run(run_name="logistic_regression_smote"):
#     mlflow.set_tag("owner", owner)

# =====================
# SMOTE (só no treino)
# =====================
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# =====================
# Modelo
# =====================
model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

model.fit(X_train_smote, y_train_smote)

# =====================
# Predições
# =====================
y_pred_train = model.predict(X_train_smote)
y_pred_test = model.predict(X_test)

# =====================
# Métricas
# =====================
train_accuracy = accuracy_score(y_train_smote, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)
test_f1 = f1_score(y_test, y_pred_test)
test_precision = precision_score(y_test, y_pred_test)
test_recall = recall_score(y_test, y_pred_test)

overfitting = train_accuracy - test_accuracy

# =====================
# Log no MLflow (quando quiser usar)
# =====================
# mlflow.log_metric("train_accuracy", train_accuracy)
# mlflow.log_metric("test_accuracy", test_accuracy)
# mlflow.log_metric("test_f1_score", test_f1)
# mlflow.log_metric("test_precision", test_precision)
# mlflow.log_metric("test_recall", test_recall)
# mlflow.log_metric("overfitting", overfitting)

# mlflow.log_param("model", "logistic_regression")
# mlflow.log_param("sampling", "SMOTE")

# =====================
# Print
# =====================
print(f"=== LOGISTIC REGRESSION (SMOTE) ===")
print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy:  {test_accuracy:.4f}")
print(f"Test F1 Score:  {test_f1:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall:    {test_recall:.4f}")
print(f"Overfitting:    {overfitting:.4f}")

=== LOGISTIC REGRESSION (SMOTE) ===
Train Accuracy: 0.8421
Test Accuracy:  0.7729
Test F1 Score:  0.6330
Test Precision: 0.5847
Test Recall:    0.6900
Overfitting:    0.0692


c:\Users\Patrick\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


| Modelo   | Precision | Recall   | F1       | Overfitting  |
| -------- | --------- | -------- | -------- | ------------ |
| Baseline | 0.70      | 0.56     | 0.62     | 0.007        |
| Balanced | 0.55      | **0.81** | **0.65** | 0.006        |
| SMOTE    | 0.58      | 0.69     | 0.63     | **0.069 ⚠️** |

Foram testadas diferentes estratégias para lidar com desbalanceamento, incluindo class_weight e SMOTE. A abordagem com class_weight='balanced' apresentou melhor desempenho, com maior recall e menor overfitting, sendo mais adequada para o problema de churn.

In [12]:
# DummyClassifier
from sklearn.dummy import DummyClassifier

treino_x, teste_x, treino_y, teste_y = train_test_split(X, y, random_state=42, stratify=y, test_size=0.25)

classificador = DummyClassifier()
classificador.fit(treino_x, treino_y)
previsoes = classificador.predict(teste_x)

acuracia = accuracy_score(teste_y, previsoes) * 100
print(f"A acurácia do dummy foi de {acuracia:.2f}%")

A acurácia do dummy foi de 73.48%
